
# Weather Dataset — Data Cleaning

## Objective

This notebook validates and cleans the district-level annual NASA POWER weather dataset used in BharatCrop Intelligence. It identifies invalid records, suspicious measurements, inconsistent geography mappings, and potential data-quality issues.

## Cleaning Strategy

- Validate dataset structure and required columns.
- Check missing values, duplicate records, and district-year keys.
- Validate numeric types and weather parameter ranges.
- Flag suspicious extreme measurements without automatically deleting plausible extremes.
- Validate temperature and precipitation consistency.
- Inspect district-to-grid mapping distances.
- Save a separate quality-flagged dataset and a cleaning report.

## Important Rules

- Preserve the source dataset.
- Do not replace missing values with zero.
- Do not remove extreme observations solely because they are extreme.
- Distinguish confirmed errors from values that need investigation.
- Document every cleaning decision.

## Input Dataset

`data/processed/weather_regional/district_weather_annual_2013_2024.csv`

## Expected Output

`data/processed/weather_regional/district_weather_annual_cleaned_flagged_2013_2024.csv`


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

input_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "district_weather_annual_2013_2024.csv"
)

output_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "district_weather_annual_cleaned_flagged_2013_2024.csv"
)

if not input_path.exists():
    raise FileNotFoundError(f"Source dataset not found: {input_path}")

weather_raw = pd.read_csv(input_path)
weather_clean = weather_raw.copy(deep=True)

print("Source exists:", input_path.exists())
print("Input shape:", weather_raw.shape)
print("Working copy shape:", weather_clean.shape)
print("Output already exists:", output_path.exists())
print("Source and working copy initially identical:",
      weather_raw.equals(weather_clean))


Source exists: True
Input shape: (7680, 24)
Working copy shape: (7680, 24)
Output already exists: False
Source and working copy initially identical: True


In [2]:

# Validate required columns and structural integrity.

required_columns = [
    "apy_geography_id",
    "State",
    "District",
    "district_latitude",
    "district_longitude",
    "coordinate_source",
    "geography_confidence",
    "nearest_grid_latitude",
    "nearest_grid_longitude",
    "nearest_grid_elevation_m",
    "nearest_grid_distance_km",
    "latitude",
    "longitude",
    "calendar_year",
    "tmin_mean_c",
    "tmin_min_c",
    "tmin_valid_days",
    "tmax_mean_c",
    "tmax_max_c",
    "tmax_valid_days",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day",
    "precipitation_valid_days",
]

missing_columns = sorted(set(required_columns) - set(weather_clean.columns))
unexpected_columns = sorted(set(weather_clean.columns) - set(required_columns))

print("Missing required columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)
print("Missing values across dataset:", int(weather_clean.isna().sum().sum()))
print("Exact duplicate rows:", int(weather_clean.duplicated().sum()))
print(
    "Duplicate district-year keys:",
    int(weather_clean.duplicated(["apy_geography_id", "calendar_year"]).sum())
)

if missing_columns:
    raise ValueError("Required columns are missing. Resolve the schema before cleaning.")

print("\nColumn data types:")
display(weather_clean.dtypes.astype(str).rename("Data_Type").to_frame())


Missing required columns: []
Unexpected columns: []
Missing values across dataset: 0
Exact duplicate rows: 0
Duplicate district-year keys: 0

Column data types:


,Data_Type
apy_geography_id,object
State,object
District,object
district_latitude,float64
district_longitude,float64
coordinate_source,object
geography_confidence,object
nearest_grid_latitude,float64
nearest_grid_longitude,float64
nearest_grid_elevation_m,float64


In [3]:

# Validate year coverage, coordinate ranges, and annual valid-day counts.

checks = {
    "Year outside 2013–2024": ~weather_clean["calendar_year"].between(2013, 2024),
    "Invalid district latitude": ~weather_clean["district_latitude"].between(-90, 90),
    "Invalid district longitude": ~weather_clean["district_longitude"].between(-180, 180),
    "Invalid grid latitude": ~weather_clean["nearest_grid_latitude"].between(-90, 90),
    "Invalid grid longitude": ~weather_clean["nearest_grid_longitude"].between(-180, 180),
    "Negative mapping distance": weather_clean["nearest_grid_distance_km"] < 0,
    "Invalid minimum-temperature valid days": ~weather_clean["tmin_valid_days"].isin([365, 366]),
    "Invalid maximum-temperature valid days": ~weather_clean["tmax_valid_days"].isin([365, 366]),
    "Invalid precipitation valid days": ~weather_clean["precipitation_valid_days"].isin([365, 366]),
}

for description, condition in checks.items():
    print(f"{description}: {int(condition.sum())} records")

print("\nObserved year range:")
print(weather_clean["calendar_year"].min(), "to",
      weather_clean["calendar_year"].max())

print("\nObserved district coordinate ranges:")
print(
    "Latitude:",
    weather_clean["district_latitude"].min(),
    "to",
    weather_clean["district_latitude"].max()
)
print(
    "Longitude:",
    weather_clean["district_longitude"].min(),
    "to",
    weather_clean["district_longitude"].max()
)


Year outside 2013–2024: 0 records
Invalid district latitude: 0 records
Invalid district longitude: 0 records
Invalid grid latitude: 0 records
Invalid grid longitude: 0 records
Negative mapping distance: 0 records
Invalid minimum-temperature valid days: 0 records
Invalid maximum-temperature valid days: 0 records
Invalid precipitation valid days: 0 records

Observed year range:
2013 to 2024

Observed district coordinate ranges:
Latitude: 6.998715229240884 to 34.51734539002072
Longitude: 69.42876395997312 to 96.88198947295628


In [4]:

# Check physical consistency of weather measurements.

weather_checks = {
    "Mean minimum temperature exceeds mean maximum temperature": (
        weather_clean["tmin_mean_c"] > weather_clean["tmax_mean_c"]
    ),
    "Annual minimum temperature exceeds annual maximum temperature": (
        weather_clean["tmin_min_c"] > weather_clean["tmax_max_c"]
    ),
    "Mean minimum temperature is below annual minimum": (
        weather_clean["tmin_mean_c"] < weather_clean["tmin_min_c"]
    ),
    "Mean maximum temperature exceeds annual maximum": (
        weather_clean["tmax_mean_c"] > weather_clean["tmax_max_c"]
    ),
    "Negative annual precipitation": (
        weather_clean["precipitation_total_mm"] < 0
    ),
    "Negative mean daily precipitation": (
        weather_clean["precipitation_mean_mm_day"] < 0
    ),
    "Negative maximum daily precipitation": (
        weather_clean["precipitation_max_mm_day"] < 0
    ),
    "Annual precipitation below mean daily precipitation": (
        weather_clean["precipitation_total_mm"]
        < weather_clean["precipitation_mean_mm_day"]
    ),
    "Mean daily precipitation exceeds maximum daily precipitation": (
        weather_clean["precipitation_mean_mm_day"]
        > weather_clean["precipitation_max_mm_day"]
    ),
}

for description, condition in weather_checks.items():
    print(f"{description}: {int(condition.sum())} records")

# Verify annual precipitation aggregation.
expected_total = (
    weather_clean["precipitation_mean_mm_day"]
    * weather_clean["precipitation_valid_days"]
)

difference = (
    weather_clean["precipitation_total_mm"] - expected_total
).abs()

print("\nPrecipitation aggregation mismatches (> 0.1 mm):",
      int((difference > 0.1).sum()))
print("Maximum absolute aggregation difference (mm):",
      round(difference.max(), 4))


Mean minimum temperature exceeds mean maximum temperature: 0 records
Annual minimum temperature exceeds annual maximum temperature: 0 records
Mean minimum temperature is below annual minimum: 0 records
Mean maximum temperature exceeds annual maximum: 0 records
Negative annual precipitation: 0 records
Negative mean daily precipitation: 0 records
Negative maximum daily precipitation: 0 records
Annual precipitation below mean daily precipitation: 0 records
Mean daily precipitation exceeds maximum daily precipitation: 0 records

Precipitation aggregation mismatches (> 0.1 mm): 0
Maximum absolute aggregation difference (mm): 0.0


In [5]:

# Add review flags without deleting or changing source measurements.

weather_clean["QC_Extreme_Annual_Min_Temperature"] = (
    weather_clean["tmin_min_c"] < -30
)

weather_clean["QC_Extreme_Annual_Max_Temperature"] = (
    weather_clean["tmax_max_c"] > 48
)

weather_clean["QC_High_Annual_Precipitation"] = (
    weather_clean["precipitation_total_mm"] > 5000
)

weather_clean["QC_Extreme_Daily_Precipitation"] = (
    weather_clean["precipitation_max_mm_day"] > 300
)

weather_clean["QC_Long_District_Grid_Distance"] = (
    weather_clean["nearest_grid_distance_km"] > 25
)

quality_flag_columns = [
    "QC_Extreme_Annual_Min_Temperature",
    "QC_Extreme_Annual_Max_Temperature",
    "QC_High_Annual_Precipitation",
    "QC_Extreme_Daily_Precipitation",
    "QC_Long_District_Grid_Distance",
]

print("Quality flag counts:")
for column in quality_flag_columns:
    print(f"{column}: {int(weather_clean[column].sum())}")

print("\nRecords with at least one review flag:",
      int(weather_clean[quality_flag_columns].any(axis=1).sum()))

print("\nReview records with extreme measurements:")
display(
    weather_clean.loc[
        weather_clean[
            [
                "QC_Extreme_Annual_Min_Temperature",
                "QC_Extreme_Annual_Max_Temperature",
                "QC_High_Annual_Precipitation",
                "QC_Extreme_Daily_Precipitation",
            ]
        ].any(axis=1),
        [
            "State", "District", "calendar_year",
            "tmin_min_c", "tmax_max_c",
            "precipitation_total_mm", "precipitation_max_mm_day",
            *quality_flag_columns,
        ],
    ].head(20)
)


Quality flag counts:
QC_Extreme_Annual_Min_Temperature: 11
QC_Extreme_Annual_Max_Temperature: 498
QC_High_Annual_Precipitation: 13
QC_Extreme_Daily_Precipitation: 11
QC_Long_District_Grid_Distance: 3396

Records with at least one review flag: 3718

Review records with extreme measurements:


,State,District,calendar_year,tmin_min_c,tmax_max_c,precipitation_total_mm,precipitation_max_mm_day,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance
134,Andhra Pradesh,Guntur,2015,13.38,48.14,823.93,40.53,False,True,False,False,False
194,Andhra Pradesh,Ntr,2015,11.95,48.05,574.03,31.08,False,True,False,False,True
206,Andhra Pradesh,Palnadu,2015,11.71,48.25,627.67,27.40,False,True,False,False,True
950,Bihar,Arwal,2015,5.44,48.41,1068.98,67.33,False,True,False,False,True
954,Bihar,Arwal,2019,2.46,48.22,962.75,71.99,False,True,False,False,True
955,Bihar,Arwal,2020,3.78,48.03,1216.99,136.96,False,True,False,False,True
959,Bihar,Arwal,2024,6.15,49.11,883.31,39.29,False,True,False,False,True
962,Bihar,Aurangabad,2015,5.44,48.41,1068.98,67.33,False,True,False,False,False
966,Bihar,Aurangabad,2019,2.46,48.22,962.75,71.99,False,True,False,False,False
967,Bihar,Aurangabad,2020,3.78,48.03,1216.99,136.96,False,True,False,False,False


In [6]:

# Summarize review flags and their overlap.

measurement_flag_columns = [
    "QC_Extreme_Annual_Min_Temperature",
    "QC_Extreme_Annual_Max_Temperature",
    "QC_High_Annual_Precipitation",
    "QC_Extreme_Daily_Precipitation",
]

print("Unique districts affected by each flag:")
for column in quality_flag_columns:
    affected = weather_clean.loc[
        weather_clean[column], "apy_geography_id"
    ].nunique()
    print(f"{column}: {affected} districts")

print("\nRecords by number of review flags:")
flag_counts_per_record = weather_clean[quality_flag_columns].sum(axis=1)
display(flag_counts_per_record.value_counts().sort_index().rename("Record_Count"))

print("\nRecords with measurement flags:",
      int(weather_clean[measurement_flag_columns].any(axis=1).sum()))

print("Records with spatial flag only:",
      int((
          weather_clean["QC_Long_District_Grid_Distance"]
          & ~weather_clean[measurement_flag_columns].any(axis=1)
      ).sum()))

print("Unique districts with spatial flag:",
      weather_clean.loc[
          weather_clean["QC_Long_District_Grid_Distance"],
          "apy_geography_id"
      ].nunique())


Unique districts affected by each flag:
QC_Extreme_Annual_Min_Temperature: 3 districts
QC_Extreme_Annual_Max_Temperature: 184 districts
QC_High_Annual_Precipitation: 9 districts
QC_Extreme_Daily_Precipitation: 9 districts
QC_Long_District_Grid_Distance: 283 districts

Records by number of review flags:


0    3962
1    3508
2     209
3       1
Name: Record_Count, dtype: int64


Records with measurement flags: 532
Records with spatial flag only: 3186
Unique districts with spatial flag: 283


In [7]:

# Save the quality-flagged dataset and a cleaning report.
# No source measurements are deleted, imputed, or overwritten.

report_path = (
    project_root
    / "reports"
    / "data_audit"
    / "weather_cleaning_report_2013_2024.csv"
)

report_path.parent.mkdir(parents=True, exist_ok=True)
output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists():
    raise FileExistsError(
        f"Output already exists; refusing to overwrite: {output_path}"
    )

# Build an auditable summary of checks and outcomes.
report_rows = [
    ("Input records", len(weather_raw), "All source records loaded"),
    ("Output records", len(weather_clean), "No records removed"),
    ("Missing values", int(weather_clean[weather_raw.columns].isna().sum().sum()),
     "No imputation performed"),
    ("Exact duplicate rows", int(weather_raw.duplicated().sum()),
     "None detected"),
    ("Duplicate district-year keys",
     int(weather_raw.duplicated(["apy_geography_id", "calendar_year"]).sum()),
     "None detected"),
    ("Records with measurement flags",
     int(weather_clean[measurement_flag_columns].any(axis=1).sum()),
     "Retained for review"),
    ("Records with spatial flag",
     int(weather_clean["QC_Long_District_Grid_Distance"].sum()),
     "Retained; mapping limitation"),
    ("Records with any review flag",
     int(weather_clean[quality_flag_columns].any(axis=1).sum()),
     "Retained; flags are not automatic error labels"),
]

cleaning_report = pd.DataFrame(
    report_rows,
    columns=["Check", "Count", "Outcome"]
)

# Save outputs.
weather_clean.to_csv(output_path, index=False)
cleaning_report.to_csv(report_path, index=False)

print("Flagged dataset saved:", output_path)
print("Cleaning report saved:", report_path)
print("Output shape:", weather_clean.shape)
print("\nCleaning report:")
display(cleaning_report)


Flagged dataset saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\district_weather_annual_cleaned_flagged_2013_2024.csv
Cleaning report saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit\weather_cleaning_report_2013_2024.csv
Output shape: (7680, 29)

Cleaning report:


,Check,Count,Outcome
0,Input records,7680,All source records loaded
1,Output records,7680,No records removed
2,Missing values,0,No imputation performed
3,Exact duplicate rows,0,None detected
4,Duplicate district-year keys,0,None detected
5,Records with measurement flags,532,Retained for review
6,Records with spatial flag,3396,Retained; mapping limitation
7,Records with any review flag,3718,Retained; flags are not automatic error labels


In [8]:

# Verify saved files and confirm original measurements were preserved.

saved_weather = pd.read_csv(output_path)
saved_report = pd.read_csv(report_path)

original_columns = weather_raw.columns.tolist()

print("Output file exists:", output_path.exists())
print("Report file exists:", report_path.exists())
print("Saved dataset shape:", saved_weather.shape)
print("Saved report shape:", saved_report.shape)

print(
    "\nOriginal measurement and source columns unchanged:",
    saved_weather[original_columns].equals(weather_raw[original_columns])
)

print(
    "All quality flags present:",
    all(column in saved_weather.columns for column in quality_flag_columns)
)

print(
    "Saved dataset record count matches source:",
    len(saved_weather) == len(weather_raw)
)

print(
    "Saved report confirms no records removed:",
    int(saved_report.loc[
        saved_report["Check"] == "Output records", "Count"
    ].iloc[0]) == len(weather_raw)
)

print("\nSaved quality-flag counts:")
display(saved_weather[quality_flag_columns].sum().rename("Flagged_Records"))


Output file exists: True
Report file exists: True
Saved dataset shape: (7680, 29)
Saved report shape: (8, 3)

Original measurement and source columns unchanged: True
All quality flags present: True
Saved dataset record count matches source: True
Saved report confirms no records removed: True

Saved quality-flag counts:


QC_Extreme_Annual_Min_Temperature      11
QC_Extreme_Annual_Max_Temperature     498
QC_High_Annual_Precipitation           13
QC_Extreme_Daily_Precipitation         11
QC_Long_District_Grid_Distance       3396
Name: Flagged_Records, dtype: int64